# 06 - Pairwise judging with Kaggle models (Task 7)

Create this as a **utility benchmark** notebook so `kaggle_benchmarks` model calls are available. Attach the saved output of notebook 05. No GPU needed. The judge runs in-kernel (via `%run`) so it uses this notebook's model access. Re-running resumes: judged pairs are skipped, failed batches are retried. Save Version when done.

In [ ]:
REPO_URL = "https://github.com/IsrarAliKhan/small-model-alignment-starter.git"  # edit if your fork lives elsewhere
%cd /kaggle/working
!git clone -q {REPO_URL} || git -C small-model-alignment-starter pull -q
%cd /kaggle/working/small-model-alignment-starter


In [ ]:
import glob

def find(pattern):
    """Locate a file/dir inside attached Kaggle inputs (outputs of earlier notebooks)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Not found: {pattern}. Attach the saved output of the notebook that produced it."
    print(pattern, "->", hits[0])
    return hits[0]

In [ ]:
import kaggle_benchmarks as kbench
JUDGE_MODEL = "google/gemini-3.6-flash"
print(JUDGE_MODEL in kbench.llms, [m for m in kbench.llms if "gemini" in m])  # pick another if unavailable
PAIRS = str(__import__("pathlib").Path(find("outputs/eval/dpo_uf_vs_sft_alpaca_pairs.jsonl")).parent)

Smoke test on 5 pairs first.

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path {PAIRS}/dpo_uf_vs_sft_alpaca_pairs.jsonl --output_path judgments/smoke_judged.jsonl --judge_model {JUDGE_MODEL} --rubric general --batch_size 5 --sleep_seconds 5 --max_retries 3 --limit 5

Full runs. HH uses the safety rubric; persona is judged twice (style-with-usefulness, and plain helpfulness/safety).

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path {PAIRS}/dpo_uf_vs_sft_alpaca_pairs.jsonl --output_path judgments/dpo_uf_vs_sft_alpaca_judged.jsonl --judge_model {JUDGE_MODEL} --rubric general --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path {PAIRS}/dpo_hh_vs_sft_hh_pairs.jsonl --output_path judgments/dpo_hh_vs_sft_hh_judged.jsonl --judge_model {JUDGE_MODEL} --rubric safety --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path {PAIRS}/lora_vs_base_persona_pairs.jsonl --output_path judgments/lora_vs_base_persona_style_judged.jsonl --judge_model {JUDGE_MODEL} --rubric persona_style --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments
%run scripts/judge_pairwise_kaggle.py --pairs_path {PAIRS}/lora_vs_base_persona_pairs.jsonl --output_path judgments/lora_vs_base_persona_helpful_judged.jsonl --judge_model {JUDGE_MODEL} --rubric safety --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

In [ ]:
import json, glob
for f in sorted(glob.glob('judgments/*_summary.json')):
    s = json.load(open(f)); print(f, {k: s[k] for k in ['n_pairs','win','loss','tie','unknown','failure','win_rate_ties_half','win_rate_excluding_ties','verdict_A','verdict_B']})